# Pseudocode to code: IIR filtering of IQ data

The first section follows the local pseudocode skill: `←` means assignment, `=` means comparison, arrays are zero-indexed, and every block is explicitly closed. The second section translates the algorithm to Python and SciPy.

```text
FUNCTION GenerateSyntheticIQ(exampleCount, sampleCount, seed)
    DECLARE X AS ARRAY OF FLOAT32
    X ← array with shape (exampleCount, 2, sampleCount)
    rng ← deterministic random generator initialized with seed

    FOR exampleIndex ← 0 TO exampleCount - 1
        phase ← 0.4 × exampleIndex
        X[exampleIndex, 0, :] ← low-frequency sine + noise
        X[exampleIndex, 1, :] ← low-frequency cosine + noise
    END FOR

    RETURN X
END FUNCTION

FUNCTION DesignButterworthIIR(cutoff, order)
    DECLARE b AS ARRAY
    DECLARE a AS ARRAY
    (b, a) ← Butterworth low-pass coefficients with cutoff and order
    RETURN (b, a)
END FUNCTION

FUNCTION ApplyIIRFilter(signal, b, a)
    DECLARE filtered AS ARRAY
    filtered ← IIR filter of signal using b and a along axis 2
    RETURN filtered
END FUNCTION

FUNCTION ComputePower(iqArray)
    DECLARE iComponent AS ARRAY
    DECLARE qComponent AS ARRAY
    iComponent ← iqArray[:, 0, :]
    qComponent ← iqArray[:, 1, :]
    RETURN mean(iComponent² + qComponent²)
END FUNCTION

X ← GenerateSyntheticIQ(exampleCount = 5, sampleCount = 1000, seed = 42)
(b, a) ← DesignButterworthIIR(cutoff = 0.1, order = 2)
XFiltered ← ApplyIIRFilter(X, b, a)
powerBefore ← ComputePower(X)
powerAfter ← ComputePower(XFiltered)

IF XFiltered.shape = X.shape THEN
    PRINT 'shape verified'
END IF

IF powerAfter < powerBefore THEN
    PRINT 'low-pass filtering reduced power'
END IF
```

In [ ]:
from pathlib import Path
import numpy as np
from scipy import signal

In [ ]:
def generate_synthetic_iq(example_count, sample_count, seed):
    rng = np.random.default_rng(seed)
    time = np.arange(sample_count, dtype=np.float32)
    iq_array = np.empty((example_count, 2, sample_count), dtype=np.float32)
    for example_index in range(example_count):
        phase = 0.4 * example_index
        low_frequency = 0.03
        high_frequency = 0.35
        noise_scale = 0.25
        iq_array[example_index, 0] = (
            np.sin(2 * np.pi * low_frequency * time + phase)
            + 0.5 * np.sin(2 * np.pi * high_frequency * time)
            + noise_scale * rng.standard_normal(sample_count)
        )
        iq_array[example_index, 1] = (
            np.cos(2 * np.pi * low_frequency * time + phase)
            + 0.5 * np.cos(2 * np.pi * high_frequency * time)
            + noise_scale * rng.standard_normal(sample_count)
        )
    return iq_array

def design_butterworth_iir(cutoff, order):
    return signal.butter(order, cutoff, btype='lowpass')

def apply_iir_filter(iq_array, b, a):
    return signal.lfilter(b, a, iq_array, axis=2).astype(np.float32)

def compute_power(iq_array):
    i_component = iq_array[:, 0, :]
    q_component = iq_array[:, 1, :]
    return float(np.mean(i_component ** 2 + q_component ** 2))

In [ ]:
N, L, SEED = 5, 1000, 42
X = generate_synthetic_iq(N, L, SEED)
b, a = design_butterworth_iir(cutoff=0.1, order=2)
X_filtered = apply_iir_filter(X, b, a)
power_before = compute_power(X)
power_after = compute_power(X_filtered)

assert X.shape == (N, 2, L)
assert X_filtered.shape == X.shape
assert X_filtered.dtype == np.float32
assert not np.allclose(X_filtered, X)
assert power_after < power_before

output_path = Path('Novoa') / 'filtered_iq_from_pseudocode.npz'
output_path.parent.mkdir(parents=True, exist_ok=True)
np.savez(output_path, X=X, X_filtered=X_filtered, b=b, a=a, seed=SEED)
print(f'Input shape: {X.shape}')
print(f'Filtered shape: {X_filtered.shape}')
print(f'Power before: {power_before:.6f}')
print(f'Power after: {power_after:.6f}')
print(f'Power ratio: {power_after / power_before:.6f}')
print(f'Saved: {output_path}')
print('PASS: pseudocode translation, shape, dtype, changed signal, power, and NPZ output verified')